# SNU AI Challenge — Full 24-Class Colab Pipeline

This notebook starts from the original 9,535 labeled training samples. It includes all 1,478 `No_ordering=True` samples whose target is `[1, 2, 3, 4]`. No result or cache from the previous 8,057-row ordering-only run is reused.

Run the cells from top to bottom using a VS Code Colab GPU kernel or Google Colab.

## 1. Goal and assumptions

The notebook extracts the raw-data bundle, validates the full 24-class split, computes frozen ConvNeXt/Qwen representations, runs full-data five-fold validation, and finally trains one scorer on all 9,535 rows. Feature-cache shards and checkpoints are persisted under `MyDrive/snu ai/`.

Use only the provided competition training data. Do not place test data in this training bundle.

## 2. Parameters

For parallel Colab runtimes, assign disjoint fold lists. Never run the same fold in two runtimes writing to the same result directory.

In [2]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
BUNDLE_PATH = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
CODE_BUNDLE_PATH = DRIVE_ROOT / "snu_colab_resident_training_code.tar"
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
DRIVE_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
DRIVE_RESULTS_ROOT = DRIVE_ROOT / "results/full_24class_resident_v2"
LOCAL_CACHE_ROOT = WORKDIR / "feature_cache/full_train_9535"

FOLDS_TO_RUN = [0, 1, 2, 3, 4]
CACHE_SHARD_SIZE = 250
TRAIN_FINAL_SCORER = True

HIDDEN_DIM = 256
VERIFIER = "transformer"
BATCH_SIZE = 128
EVALUATION_BATCH_SIZE = 256
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TOP_H = 5
TOP_M = 8
ALPHA, BETA, GAMMA, DELTA = 0.1, 1.0, 1.0, 1.0
SEED = 42

assert FOLDS_TO_RUN and all(fold in range(5) for fold in FOLDS_TO_RUN)
assert len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))
assert CACHE_SHARD_SIZE > 0


## 3. Mount Drive and install dependencies

In [3]:
try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
else:
    print("google.colab is unavailable; configured paths will be used directly.")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_SHARDS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [4]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "transformers==5.13.0",
        "accelerate==1.14.0",
        "pillow>=10",
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', 'transformers==5.13.0', 'accelerate==1.14.0', 'pillow>=10'], returncode=0)

## 4. Extract the raw training bundle locally

The bundle must contain `snu_ordering/`, `train.csv`, all 9,535 training image folders, and `full_folds_5.csv`. Files are extracted to `/content` for fast access.

In [5]:
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Training bundle not found: {BUNDLE_PATH}")
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Optimized code bundle not found: {CODE_BUNDLE_PATH}")

WORKDIR.mkdir(parents=True, exist_ok=True)
bundle_marker = WORKDIR / "snuaichallenge_data/train.csv"
if not bundle_marker.is_file():
    subprocess.run(
        ["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR)],
        check=True,
    )

subprocess.run(
    ["tar", "-xf", str(CODE_BUNDLE_PATH), "-C", str(WORKDIR)],
    check=True,
)
for module_name in list(sys.modules):
    if module_name == "snu_ordering" or module_name.startswith("snu_ordering."):
        del sys.modules[module_name]
import importlib
importlib.invalidate_caches()

print(f"Project root: {WORKDIR}")
print(f"Persistent cache shards: {DRIVE_CACHE_SHARDS}")
print(f"Persistent results: {DRIVE_RESULTS_ROOT}")


Project root: /content/snu_ai
Persistent cache shards: /content/drive/MyDrive/snu ai/cache/full_train_9535_shards
Persistent results: /content/drive/MyDrive/snu ai/results/full_24class_resident_v2


## 5. Validate GPU, data, and the full 24-class split

In [6]:
import json
import random
from collections import Counter

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")

sys.path.insert(0, str(WORKDIR))
from snu_ordering.data import load_samples

IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
FULL_FOLDS_CSV = WORKDIR / "snuaichallenge_data/splits/full_folds_5.csv"
samples = load_samples(
    FULL_FOLDS_CSV,
    IMAGE_ROOT,
    require_labels=True,
    validate_images=True,
)

answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
identity = (1, 2, 3, 4)
no_ordering_count = sum(sample.no_ordering is True for sample in samples)
identity_count = answer_counts[identity]

assert len(samples) == 9535
assert len(answer_counts) == 24
assert identity_count == 1478
assert no_ordering_count == 1478
assert all(1905 <= fold_counts[fold] <= 1909 for fold in range(5))
assert all(sample.fold is not None for sample in samples)

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
print("rows / classes / identity:", len(samples), len(answer_counts), identity_count)
print("fold counts:", dict(sorted(fold_counts.items())))


torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB
CUDA capability: (8, 0)
rows / classes / identity: 9535 24 1478
fold counts: {0: 1907, 1: 1907, 2: 1909, 3: 1907, 4: 1905}


## 6. Restore persisted feature-cache shards

Cache shards contain 250 samples each. On a new Colab runtime, completed shards are extracted to local storage and only missing samples are encoded.

In [7]:
import tarfile

LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
persisted_relative_paths = set()
cache_shards = sorted(DRIVE_CACHE_SHARDS.glob("cache_*.tar"))
for shard_path in cache_shards:
    with tarfile.open(shard_path, "r") as archive:
        members = [member for member in archive.getmembers() if member.isfile()]
        persisted_relative_paths.update(member.name for member in members)
        archive.extractall(LOCAL_CACHE_ROOT)

print(f"restored shards: {len(cache_shards)}")
print(f"restored cache files: {len(persisted_relative_paths)}")


/tmp/ipykernel_824/1588790026.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(LOCAL_CACHE_ROOT)


restored shards: 39
restored cache files: 9535


## 7. Cache all 9,535 frozen representations

Qwen is pinned to revision `37eefda6797047d24d5d08184aea3e097f3f8a50`. Ampere-or-newer GPUs use BF16; T4 uses FP16. The cache step is resumable from Drive shards.

In [9]:
import gc
import shutil
import time

from snu_ordering.data import load_sample_images
from snu_ordering.features import ConvNeXtTinyFeatureExtractor
from snu_ordering.qwen_representations import (
    DEFAULT_MAX_PIXELS,
    DEFAULT_MIN_PIXELS,
    FrozenQwenRepresentationExtractor,
    QWEN_MODEL_ID,
    QWEN_MODEL_REVISION,
    build_structured_representation,
    cache_path_for,
    load_representation,
    representation_fingerprint,
    save_representation,
)
from snu_ordering.text import chunk_caption

existing_indices = [int(path.stem.split("_")[-1]) for path in cache_shards]
next_shard_index = max(existing_indices, default=-1) + 1

def persist_cache_paths(paths):
    global next_shard_index
    if not paths:
        return
    local_tar = WORKDIR / f"cache_{next_shard_index:04d}.tar"
    final_tar = DRIVE_CACHE_SHARDS / local_tar.name
    temporary_tar = DRIVE_CACHE_SHARDS / f"{local_tar.name}.tmp"
    with tarfile.open(local_tar, "w") as archive:
        for path in paths:
            archive.add(path, arcname=path.relative_to(LOCAL_CACHE_ROOT).as_posix())
    shutil.copy2(local_tar, temporary_tar)
    temporary_tar.replace(final_tar)
    local_tar.unlink()
    print(f"persisted {len(paths)} files -> {final_tar.name}", flush=True)
    next_shard_index += 1

pending = []
unpersisted_existing = []
for sample in samples:
    chunks = chunk_caption(sample.caption)
    fingerprint = representation_fingerprint(
        sample,
        chunks,
        model_id=QWEN_MODEL_ID,
        revision=QWEN_MODEL_REVISION,
        min_pixels=DEFAULT_MIN_PIXELS,
        max_pixels=DEFAULT_MAX_PIXELS,
    )
    cache_path = cache_path_for(LOCAL_CACHE_ROOT, sample.sample_id)
    valid = False
    if cache_path.is_file():
        try:
            cached = load_representation(cache_path)
            hidden_size = cached.full_caption_feature.shape[0]
            valid = (
                cached.sample_id == sample.sample_id
                and cached.full_caption == chunks["full_caption"]
                and cached.event_units == tuple(chunks["event_units"])
                and cached.convnext_frame_features.ndim == 2
                and cached.convnext_frame_features.shape[0] == 4
                and cached.qwen_frame_features.shape == (4, hidden_size)
                and cached.event_unit_features.shape
                == (len(chunks["event_units"]), hidden_size)
            )
        except (ValueError, RuntimeError, EOFError):
            valid = False
    if valid:
        relative_name = cache_path.relative_to(LOCAL_CACHE_ROOT).as_posix()
        if relative_name not in persisted_relative_paths:
            unpersisted_existing.append(cache_path)
    else:
        pending.append((sample, chunks, fingerprint, cache_path))

for start in range(0, len(unpersisted_existing), CACHE_SHARD_SIZE):
    persist_cache_paths(unpersisted_existing[start:start + CACHE_SHARD_SIZE])

print(f"valid cached: {len(samples) - len(pending)}; pending: {len(pending)}")
if pending:
    capability_major = torch.cuda.get_device_capability(0)[0]
    qwen_dtype = torch.bfloat16 if capability_major >= 8 else torch.float16
    torch.cuda.reset_peak_memory_stats()
    convnext = ConvNeXtTinyFeatureExtractor(device="auto")
    qwen = FrozenQwenRepresentationExtractor(
        model_id=QWEN_MODEL_ID,
        revision=QWEN_MODEL_REVISION,
        device_map="auto",
        min_pixels=DEFAULT_MIN_PIXELS,
        max_pixels=DEFAULT_MAX_PIXELS,
        dtype=qwen_dtype,
    )
    shard_paths = []
    started_at = time.monotonic()
    for index, (sample, chunks, fingerprint, cache_path) in enumerate(pending, 1):
        frames = load_sample_images(sample)
        try:
            convnext_features = convnext.extract_frame_features(frames).global_features
            representation = build_structured_representation(
                sample=sample,
                frames=frames,
                convnext_frame_features=convnext_features,
                chunks=chunks,
                provider=qwen,
                fingerprint=fingerprint,
            )
            save_representation(cache_path, representation)
            shard_paths.append(cache_path)
        finally:
            for frame in frames:
                frame.close()
        if len(shard_paths) >= CACHE_SHARD_SIZE:
            persist_cache_paths(shard_paths)
            shard_paths = []
        if index % 10 == 0 or index == len(pending):
            elapsed = time.monotonic() - started_at
            print(f"[{index}/{len(pending)}] {elapsed / index:.2f} sec/sample", flush=True)
    persist_cache_paths(shard_paths)
    print("peak CUDA MB:", torch.cuda.max_memory_allocated() / 1024**2)
    del qwen, convnext
    gc.collect()
    torch.cuda.empty_cache()

final_cache_count = sum(1 for _ in LOCAL_CACHE_ROOT.rglob("*.pt"))
assert final_cache_count == 9535, final_cache_count
cache_manifest = {
    "samples": final_cache_count,
    "model_id": QWEN_MODEL_ID,
    "revision": QWEN_MODEL_REVISION,
    "min_pixels": DEFAULT_MIN_PIXELS,
    "max_pixels": DEFAULT_MAX_PIXELS,
}
with (DRIVE_CACHE_SHARDS / "manifest.json").open("w", encoding="utf-8") as file:
    json.dump(cache_manifest, file, indent=2)
    file.write("\n")
print(cache_manifest)


valid cached: 9535; pending: 0
{'samples': 9535, 'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'revision': '37eefda6797047d24d5d08184aea3e097f3f8a50', 'min_pixels': 100352, 'max_pixels': 200704}


## 8. Define full-data five-fold training

Every fold contains all 24 classes. Checkpoints and summaries are written directly to Drive.

In [10]:
from snu_ordering.structured_scorer import LightweightStructuredScorer
from snu_ordering.structured_training import (
    StructuredTrainingConfig,
    prepare_structured_samples,
    train_prepared_structured_scorer,
)

DEVICE = torch.device("cuda")
torch.set_float32_matmul_precision("high")
training_config = StructuredTrainingConfig(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    evaluation_batch_size=EVALUATION_BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=SEED,
    top_h=TOP_H,
    top_m=TOP_M,
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA,
    delta=DELTA,
    use_amp=True,
)

prepared_samples = prepare_structured_samples(
    samples, LOCAL_CACHE_ROOT, DEVICE, top_h=TOP_H
)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def run_fold(fold):
    output_dir = DRIVE_RESULTS_ROOT / f"fold_{fold}"
    if (output_dir / "training_summary.json").is_file():
        print(f"fold {fold}: already completed, skipping")
        return
    train_samples = [item for item in prepared_samples if item.sample.fold != fold]
    validation_samples = [item for item in prepared_samples if item.sample.fold == fold]
    assert {item.sample.rank_label for item in train_samples} == set(answer_counts)
    assert {item.sample.rank_label for item in validation_samples} == set(answer_counts)
    seed_everything(SEED)
    scorer = LightweightStructuredScorer(
        hidden_dim=HIDDEN_DIM,
        verifier=VERIFIER,
    ).to(DEVICE)
    print(f"fold {fold}: train={len(train_samples)}, validation={len(validation_samples)}")
    train_prepared_structured_scorer(
        scorer,
        train_samples,
        validation_samples,
        output_dir,
        training_config,
    )
    del scorer
    gc.collect()
    torch.cuda.empty_cache()


prepared candidates: 500/9535
prepared candidates: 1000/9535
prepared candidates: 1500/9535
prepared candidates: 2000/9535
prepared candidates: 2500/9535
prepared candidates: 3000/9535
prepared candidates: 3500/9535
prepared candidates: 4000/9535
prepared candidates: 4500/9535
prepared candidates: 5000/9535
prepared candidates: 5500/9535
prepared candidates: 6000/9535
prepared candidates: 6500/9535
prepared candidates: 7000/9535
prepared candidates: 7500/9535
prepared candidates: 8000/9535
prepared candidates: 8500/9535
prepared candidates: 9000/9535
prepared candidates: 9500/9535
prepared candidates: 9535/9535
resident training data: features=1.13 GiB, candidate_plans=0.35 GiB on cuda


## 9. Run selected folds

This cell is long-running. To parallelize, use separate runtimes with disjoint `FOLDS_TO_RUN` values.

In [11]:
for fold in FOLDS_TO_RUN:
    run_fold(fold)


fold 0: train=7628, validation=1907


/content/snu_ai/snu_ordering/structured_training.py:376: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_fp16_scaler)


{"epoch": 1, "train_loss": 2.9053740868970994, "samples": 1907, "exact_match_accuracy": 0.16360776088096488, "lightweight_top8_recall": 0.6858940744625066}
{"epoch": 2, "train_loss": 2.7657485102006887, "samples": 1907, "exact_match_accuracy": 0.19716832721552177, "lightweight_top8_recall": 0.7079181961195595}
{"epoch": 3, "train_loss": 2.5739306609669343, "samples": 1907, "exact_match_accuracy": 0.22863135815416885, "lightweight_top8_recall": 0.7184058730991085}
{"epoch": 4, "train_loss": 2.405767173997132, "samples": 1907, "exact_match_accuracy": 0.25222863135815415, "lightweight_top8_recall": 0.7341373885684321}
{"epoch": 5, "train_loss": 2.2524000741951617, "samples": 1907, "exact_match_accuracy": 0.254326166754064, "lightweight_top8_recall": 0.7220765600419508}
{"epoch": 6, "train_loss": 2.1313641908347574, "samples": 1907, "exact_match_accuracy": 0.2679601468274777, "lightweight_top8_recall": 0.7168327215521761}
{"epoch": 7, "train_loss": 1.9886462013322643, "samples": 1907, "exa

## 10. Aggregate five-fold results and select the final epoch count

In [12]:
import statistics

fold_records = []
for fold in range(5):
    summary_path = DRIVE_RESULTS_ROOT / f"fold_{fold}/training_summary.json"
    if not summary_path.is_file():
        continue
    with summary_path.open("r", encoding="utf-8") as file:
        summary = json.load(file)
    best = max(summary["history"], key=lambda row: row["exact_match_accuracy"])
    fold_records.append({
        "fold": fold,
        "best_epoch": best["epoch"],
        "samples": best["samples"],
        "exact_match_accuracy": best["exact_match_accuracy"],
        "lightweight_top8_recall": best.get("lightweight_top8_recall"),
        "epochs_completed": summary["epochs_completed"],
    })

aggregate = {"folds": fold_records, "completed_fold_count": len(fold_records)}
if len(fold_records) == 5:
    total_validation_samples = sum(row["samples"] for row in fold_records)
    aggregate["total_validation_samples"] = total_validation_samples
    aggregate["weighted_exact_match_accuracy"] = sum(
        row["samples"] * row["exact_match_accuracy"] for row in fold_records
    ) / total_validation_samples
    selected_final_epochs = max(
        1, int(round(statistics.median(row["best_epoch"] for row in fold_records)))
    )
    aggregate["selected_final_epochs"] = selected_final_epochs
else:
    selected_final_epochs = None

with (DRIVE_RESULTS_ROOT / "five_fold_summary.json").open("w", encoding="utf-8") as file:
    json.dump(aggregate, file, indent=2)
    file.write("\n")
print(json.dumps(aggregate, indent=2))


{
  "folds": [
    {
      "fold": 0,
      "best_epoch": 11,
      "samples": 1907,
      "exact_match_accuracy": 0.31253277399056106,
      "lightweight_top8_recall": 0.7079181961195595,
      "epochs_completed": 15
    },
    {
      "fold": 1,
      "best_epoch": 15,
      "samples": 1907,
      "exact_match_accuracy": 0.33927635028841113,
      "lightweight_top8_recall": 0.7430519140010488,
      "epochs_completed": 19
    },
    {
      "fold": 2,
      "best_epoch": 10,
      "samples": 1909,
      "exact_match_accuracy": 0.3095861707700367,
      "lightweight_top8_recall": 0.7187008905185961,
      "epochs_completed": 14
    },
    {
      "fold": 3,
      "best_epoch": 17,
      "samples": 1907,
      "exact_match_accuracy": 0.3188253801782905,
      "lightweight_top8_recall": 0.7031987414787625,
      "epochs_completed": 20
    },
    {
      "fold": 4,
      "best_epoch": 8,
      "samples": 1905,
      "exact_match_accuracy": 0.30971128608923887,
      "lightweight_top8_rec

## 11. Train one final scorer on all 9,535 rows

The epoch count is the median best epoch from full-data five-fold validation. This produces one final model, not an ensemble.

In [13]:
from dataclasses import replace

if TRAIN_FINAL_SCORER:
    if selected_final_epochs is None:
        raise RuntimeError("All five folds must finish before final training.")
    final_output_dir = DRIVE_RESULTS_ROOT / "final_scorer"
    final_summary_path = final_output_dir / "training_summary.json"
    if final_summary_path.is_file():
        print(f"final scorer already completed: {final_output_dir}")
    else:
        seed_everything(SEED)
        final_scorer = LightweightStructuredScorer(
            hidden_dim=HIDDEN_DIM,
            verifier=VERIFIER,
        ).to(DEVICE)
        final_config = replace(
            training_config,
            max_epochs=selected_final_epochs,
            patience=selected_final_epochs,
        )
        print(f"final training: samples={len(samples)}, epochs={selected_final_epochs}")
        train_prepared_structured_scorer(
            final_scorer,
            prepared_samples,
            [],
            final_output_dir,
            final_config,
        )
        del final_scorer
        gc.collect()
        torch.cuda.empty_cache()


final training: samples=9535, epochs=11
{"epoch": 1, "train_loss": 2.8845417667322906}
{"epoch": 2, "train_loss": 2.6664072865644672}
{"epoch": 3, "train_loss": 2.454842250710954}
{"epoch": 4, "train_loss": 2.286034876973953}
{"epoch": 5, "train_loss": 2.177148800366326}
{"epoch": 6, "train_loss": 2.038045971732397}
{"epoch": 7, "train_loss": 1.9169438358996012}
{"epoch": 8, "train_loss": 1.793425584126969}
{"epoch": 9, "train_loss": 1.6958753736342944}
{"epoch": 10, "train_loss": 1.6084990079576706}
{"epoch": 11, "train_loss": 1.5272639597031104}


## 12. Checks and next steps

Expected persistent outputs:

- `MyDrive/snu ai/cache/full_train_9535_shards/`: resumable frozen feature shards
- `MyDrive/snu ai/results/full_24class_resident_v2/fold_*/best.pt`: five validation checkpoints
- `MyDrive/snu ai/results/full_24class_resident_v2/five_fold_summary.json`: full 24-class CV result
- `MyDrive/snu ai/results/full_24class_resident_v2/final_scorer/final.pt`: single final scorer trained on all 9,535 rows

The final inference pipeline must still be validated offline on one RTX 3090 and must not combine the five fold checkpoints.